In [1]:
import glob
import logging
import os
import warnings
from time import time

import h5py
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from joblib import Parallel, delayed
from scipy.optimize import OptimizeWarning, minimize
from scipy.optimize._numdiff import approx_derivative
from statsmodels.robust import scale
from statsmodels.robust.norms import RobustNorm, TukeyBiweight

In [2]:
%matplotlib inline

## Function definitions

In [3]:
from dev import single_exp, double_exp, bright, single_exp_jac, double_exp_jac, bright_jac
from dev import single_exp_with_jac, double_exp_with_jac, bright_with_jac
from dev import single_exp_jax, double_exp_jax, bright_jax, TukeyBiweight_jax
from dev import nonlinear_fit

#### load data 

In [4]:
def get_valid_corrected_f(path):
    dr = path.split("/")[-1]
    data = []
    with h5py.File(f"{path}/{dr}_data.h5") as f:
        for k in f['planes'].keys():
            data.append(f[f'planes/{k}/corrected_f'][f[f'planes/{k}/valid_roi_inds'][:]])
    return data

## Data 

In [5]:
dirs = glob.glob("/data/test-dataset-for-dff_multiplane-ophys_02/*/*")
dirs

['/data/test-dataset-for-dff_multiplane-ophys_02/lamf_slc32a1_oi1/804670',
 '/data/test-dataset-for-dff_multiplane-ophys_02/lamf_slc32a1_oi1/775682',
 '/data/test-dataset-for-dff_multiplane-ophys_02/lamf_slc32a1_oi1/782149',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/753562',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/729088',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/758265',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/753561',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi4/724567',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/755212',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/759075',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/726433',
 '/data/test-dataset-for-dff_multiplane-ophys_02/snap25_oi4_dox/747443',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi1/757436',
 '/data/test-dataset-for-dff_multiplane-ophys_02/slc32a1_oi1/69

In [6]:
traces = get_valid_corrected_f(dirs[0])

In [7]:
[t.shape for t in traces]

[(56, 48374),
 (46, 48374),
 (73, 48374),
 (73, 48374),
 (62, 48374),
 (56, 48374),
 (41, 48374),
 (49, 48374),
 (59, 48403),
 (46, 48403),
 (71, 48403),
 (75, 48403),
 (65, 48403),
 (57, 48403),
 (39, 48403),
 (53, 48403),
 (57, 48427),
 (48, 48427),
 (67, 48427),
 (73, 48427),
 (65, 48427),
 (61, 48427),
 (41, 48427),
 (48, 48427),
 (55, 48390),
 (46, 48390),
 (65, 48390),
 (76, 48390),
 (63, 48390),
 (56, 48390),
 (40, 48390),
 (48, 48390),
 (60, 48373),
 (42, 48373),
 (65, 48373),
 (76, 48373),
 (66, 48373),
 (58, 48373),
 (40, 48373),
 (52, 48373),
 (58, 48366),
 (44, 48366),
 (66, 48366),
 (75, 48366),
 (61, 48366),
 (54, 48366),
 (42, 48366),
 (51, 48366)]

In [8]:
trace = traces[0][0]

In [9]:
frame_rate = 10.63  # looked up manually from session.json of multiplane-ophys_804670_2025-09-24_09-30-56_processed_2025-10-10_22-28-44
timestamps = np.arange(len(trace)) / frame_rate

In [10]:
single_exp_init = [trace[-1000:].mean(), 0.35, 3600]
double_exp_init = [trace[-1000:].mean(), 0.35, 0.2, 3600, 240]
bright_init = [trace[-1000:].mean(), 0.35, 0.2, 0.1, 0.1, 3600, 240, 50, 2000]

single_exp_bounds = [(0, np.inf)] * 2 + [(300, np.inf)]
double_exp_bounds = [(0, np.inf)] * 3 + [(300, np.inf), (1, 1200)]
bright_bounds = [(0, np.inf)] * 5 + [(300, np.inf), (1, 1200), (1, 180), (60, np.inf)]

## bounded robust regression (Tukey) 

In [11]:
for model, jac, start_params, bounds in (
    (single_exp, single_exp_jac, single_exp_init, single_exp_bounds),
    (double_exp, double_exp_jac, double_exp_init, double_exp_bounds),
    (bright, bright_jac, bright_init, bright_bounds),
):
    print("\n" + model.__name__)
    for method in ("Nelder-Mead", "L-BFGS-B_noJac", "L-BFGS-B"):
        optimizer = method[:-6] if method[-5:]=="noJac" else method
        tic = -time()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            warnings.simplefilter("ignore", category=OptimizeWarning)
            F0, res = nonlinear_fit(trace, timestamps, model, start_params, bounds=bounds,
                                         M=TukeyBiweight(3),
                                         jac=jac if method=="L-BFGS-B" else None,
                                         optimizer=optimizer,
                                         optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10) if optimizer=="L-BFGS-B" else None)
        tic += time()
        with np.printoptions(precision=2, suppress=False, linewidth=120):
            print(f"{method:14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)


single_exp
Nelder-Mead     Time= 0.8263s  Loss=29155.95 [ 1030.81     0.   18031.9 ] Optimization terminated successfully.
L-BFGS-B_noJac  Time= 0.0797s  Loss=29160.82 [1030.83    0.   3599.02] CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL
L-BFGS-B        Time= 0.0411s  Loss=29160.84 [1030.83    0.   3599.02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp
Nelder-Mead     Time= 2.1895s  Loss=27751.63 [1.02e+03 4.72e-15 5.00e-01 1.92e+04 1.03e+02] Optimization terminated successfully.
L-BFGS-B_noJac  Time= 1.8886s  Loss=27753.58 [1.02e+03 0.00e+00 5.00e-01 3.60e+03 1.03e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
L-BFGS-B        Time= 0.6681s  Loss=27753.59 [1.02e+03 0.00e+00 5.00e-01 3.60e+03 1.03e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright
Nelder-Mead     Time=56.6720s  Loss=21633.56 [3.61e+02 3.18e+04 1.65e+04 2.82e+05 1.00e+00 2.96e+03 2.65e+02 1.47e+02 1.15e+07] Optimization terminated successfully.
L-BFGS-B_noJac  Time=18.44

#### JAX autograd

In [13]:
for dtype in (jnp.float64, jnp.float32):
    print(f"\n\n\033[1m{dtype.dtype}\033[0m")
    for model, start_params, bounds in (
        (single_exp_jax, single_exp_init, single_exp_bounds),
        (double_exp_jax, double_exp_init, double_exp_bounds),
        (bright_jax, bright_init, bright_bounds),
    ):
        print("\n" + model.__name__)
        for optimizer in ("L-BFGS-B",):
            tic = -time()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                warnings.simplefilter("ignore", category=OptimizeWarning)
                F0, res = nonlinear_fit(trace, timestamps, model, start_params, bounds=bounds, optimizer=optimizer,
                                        M=TukeyBiweight_jax(3),
                                        optimizer_options=dict(maxiter=20000, ftol=1e-12, gtol=1e-10),
                                        backend="jax",
                                        dtype=dtype)
            tic += time()
            with np.printoptions(precision=2, suppress=False, linewidth=120):
                print(f"{optimizer:14}  Time={tic:7.4f}s  Loss={res.fun/trace.size*res.sigma**2:.2f}", res.x, res.message)



float64

single_exp_jax
L-BFGS-B        Time= 0.8754s  Loss=29160.81 [1030.83    0.   3599.02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_jax
L-BFGS-B        Time= 0.7105s  Loss=27753.56 [1.02e+03 0.00e+00 5.00e-01 3.60e+03 1.03e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_jax
L-BFGS-B        Time= 2.7137s  Loss=21568.28 [6.43e+02 2.77e+00 6.43e+01 2.43e+02 9.94e-01 3.53e+03 1.30e+02 1.70e+01 1.89e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH


float32

single_exp_jax
L-BFGS-B        Time= 0.9399s  Loss=29160.29 [1030.9     0.   3599.02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

double_exp_jax
L-BFGS-B        Time= 0.6629s  Loss=28265.45 [1.01e+03 0.00e+00 2.90e-01 3.60e+03 2.23e+02] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH

bright_jax
L-BFGS-B        Time= 0.7999s  Loss=22942.49 [1.06e+03 5.78e-01 4.31e+00 0.00e+00 7.51e-01 3.60e+03 1.89e+02 5.32e+01 2.00e+03] CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR